In [1]:
import os
import pandas as pd
from scipy.stats import chi2_contingency
from statsmodels.stats.contingency_tables import Table
import math
from collections import defaultdict
from typing import Tuple

In [2]:
sp18_ext_path = "../resources/developed/sp18/external_effect"

res_outdir = "../out/external_effect"

outcsv_args = {
    "sep": "\t",
    "index": True,
    "header": True
}

# Study 1

Compounds built from non-unique constituent combinations (regardless of the linker) appearing only in the Sg form and not undergoing the IntPE whose N1 mostly attach a non-pluralic linker in the Sg form

vs

Compounds built from non-unique constituent combinations (regardless of the linker) appearing only in the Sg form and not undergoing the IntPE built from constituent combinations that mostly insert a non-pluralic linker in the Sg form

In [154]:
# Compounds built from non-unique constituent combinations (regardless 
# of the linker) appearing only in the Sg form and not undergoing the IntPE
# whose N1 mostly attach the pluralic linker in the Sg form
# (N=1560, UniqN=842, UniqCN=590, UniqN1N=37)
ic_alt_nipe_sgl_n1 = pd.read_csv(
    os.path.join(sp18_ext_path, "ic_alt_nipe_sgl_n1.tsv"),
    sep="\t",
    header=0,
    index_col="const_comb",
    # all str, WC int, is linker pluralic bool
    dtype=defaultdict(
        lambda: str,
        wc="int",
        is_linker_pl="bool"
    ),
    quoting=3   # = csv.QUOTE_NONE, separate by tabs only
)

ic_alt_nipe_sgl_n1.sample(5)

,gecodb,n1,n2,n1_pl_marker,n1_sem,n2_num,is_linker_pl,wc,context,context_id
const_comb,,,,,,,,,,
kind+prozess,kind_prozess,kind,prozess,_+er_,ind,pl,False,11,"""""""Da es sich trotz aller Einfacheit um höhere...",d56de1c9-75b3-49b8-b316-e529aee63f3a
person+schaden,person_+en_schaden,person,schaden,_+en_,ind,sg,True,13987,- Das weiß jeder Tierhalter : Ein Hund auf der...,cf52b9d1-bb02-4d35-a4ea-ecf37cda66e2
apfel+dieb,apfel_+=_dieb,apfel,dieb,_+=_,ind,pl,True,0,"""""""Vom Himmel fallen mehr Bomben als der gebra...",7d161d2b-b0e2-4a88-b0a7-3f334246f68c
bett+platz,bett_platz,bett,platz,_+en_,ind,pl,False,208,"""""""Der Fairness halber sei aber gesagt , dass ...",204b7576-0b60-4373-be0e-0ea438976e72
person+konzept,person_+en_konzept,person,konzept,_+en_,ind,sg,True,7,Die Unterstellung der individuellen Nutzenmaxi...,c74f104b-f2d4-4eb2-8b66-31d65a9da532


In [183]:
def calc_pll_shares(
    sp18_sample: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    # calculate absolute counts and shares of compounds 
    # with pluralic linkers in Sg vs. Pl forms

    sp18_sample_pll_counts_data = {}
    for const_comb, group in sp18_sample.groupby("const_comb"):
        for num in ["sg", "pl"]:
            sp18_sample_pll_counts_data[(const_comb, f"num_{num}")] = {
                "l_sg": len(
                    group[(group["n2_num"] == num) &
                    (~group["is_linker_pl"])]
                ),
                "l_pl": len(
                    group[(group["n2_num"] == num) &
                    (group["is_linker_pl"])]
                )
            }

    sp18_sample_pll_counts = pd.DataFrame(sp18_sample_pll_counts_data).T
    sp18_sample_pll_counts["pl_share"] = sp18_sample_pll_counts["l_pl"] / (
        sp18_sample_pll_counts["l_sg"] + sp18_sample_pll_counts["l_pl"]
    )

    sp18_sample_pll_shares = sp18_sample_pll_counts \
        .unstack()["pl_share"]  \
        [["num_sg", "num_pl"]]

    return sp18_sample_pll_counts, sp18_sample_pll_shares

In [5]:
(
    ic_alt_nipe_sgl_n1_pll_counts,
    ic_alt_nipe_sgl_n1_pll_shares
) = calc_pll_shares(ic_alt_nipe_sgl_n1)

In [10]:
ic_alt_nipe_sgl_n1_pll_counts.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sgl/pll_counts_n1.tsv"),
    **outcsv_args
)
# (UniqCN=590, UniqN1N=37),
# each constituent combination has 2 rows,
# one for Sg and one for Pl
ic_alt_nipe_sgl_n1_pll_counts

l_sg  l_pl  pl_share
apfel+bauer      num_sg     1     0       0.0
                 num_pl     0     1       1.0
apfel+baum       num_sg     2     0       0.0
                 num_pl     2     3       0.6
apfel+dieb       num_sg     0     1       1.0
...                       ...   ...       ...
zitat+verwertung num_pl     1     0       0.0
zitat+zitat      num_sg     0     1       1.0
                 num_pl     1     0       0.0
zitat+zweck      num_sg     2     0       0.0
                 num_pl     1     0       0.0

[1180 rows x 3 columns]

In [8]:
ic_alt_nipe_sgl_n1_pll_shares.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sgl/pll_shares_n1.tsv"),
    **outcsv_args
)
# (UniqCN=590, UniqN1N=37)
ic_alt_nipe_sgl_n1_pll_shares

,num_sg,num_pl
apfel+bauer,0.0,1.0
apfel+baum,0.0,0.6
apfel+dieb,1.0,1.0
apfel+hof,0.0,0.0
apfel+kopf,0.0,1.0
...,...,...
zitat+tag,0.0,0.0
zitat+text,1.0,1.0
zitat+verwertung,1.0,0.0
zitat+zitat,1.0,0.0


In [9]:
ic_alt_nipe_sgl_n1_pll_shares_desc = ic_alt_nipe_sgl_n1_pll_shares.describe()
ic_alt_nipe_sgl_n1_pll_shares_desc.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sgl/pll_shares_desc_n1.tsv"),
    **outcsv_args
)
ic_alt_nipe_sgl_n1_pll_shares_desc

,num_sg,num_pl
count,590.000000,590.000000
mean,0.340847,0.394802
std,0.429162,0.457406
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,1.000000,1.000000
max,1.000000,1.000000


In [ ]:
def sp18_orig_analysis(
    sp18_sample_pll_counts: pd.DataFrame
) -> Tuple[pd.Series, pd.Series, pd.Series]:

    # Methodology from the original [schaefer_pankratz_2018:26] paper:
    # 1. Create contingency table:
    # n2_num (Sg vs. Pl) x n1_pl_marker (zero vs. pluralic linker).
    # 2. Perform the chi-squared test of independence over the table
    # to assess the statistical significance of the association between
    # the Pl form of N2 and the presence of a pluralic linker in N1.
    # 3. Perform the Cramer's V test to assess the strength
    # of the association.
    # 4. Inspect symmetrical standardized residuals to identify
    # the cells that contribute to the association, and in which direction.

    # 1. Create contingency table replicating [schaefer_pankratz_2018:26]
    contingency_table = sp18_sample_pll_counts \
        .groupby(level=1)[["l_sg", "l_pl"]].sum()   \
        .T  \
        # [["num_sg", "num_pl"]]

    # 2. Perform the chi-squared test.

    # even though I didn't find the original R code
    # of [schaefer_pankratz_2018] for this experiment,
    # the `chi2_contingency` function from `scipy.stats`
    # produced the same results as in the paper
    # on the same contingency table, so we can be
    # confident that we are replicating the same analysis
    chi2_res = chi2_contingency(contingency_table)

    # 3. Perform the Cramer's V test to assess 
    # the strength of the association.

    # the original R code for Cramer's V is available in
    # linkingelements/Data/R/functions.R, and is as follows:
    # 
    #   cramer.v <- function(chisq) {
    #       unname(sqrt(chisq$statistic/(sum(chisq$observed)*min(dim(chisq$observed)-1))))
    #   }
    #
    # we will reconstruct it in Python:
    #   dim(chisq$observed) -> ic_alt_nipe_sg_l_ct.shape
    #
    #   min(dim(chisq$observed)-1) -> min(
    #       ic_alt_nipe_sg_l_ct.shape[0] - 1,
    #       ic_alt_nipe_sg_l_ct.shape[1] - 1
    #   )
    # 
    #   sum(chisq$observed) -> ic_alt_nipe_sg_l_ct.values.sum()
    #
    #   unname -> [irrelevant for Python]
    def cramer_v() -> float:
        return math.sqrt(
            chi2_res.statistic / (
                contingency_table.values.sum() *
                min(
                    contingency_table.shape[0] - 1,
                    contingency_table.shape[1] - 1
                )
            )
        )

    cramer_v_val = cramer_v()

    # the original R code for Cohen's W is available in
    # linkingelements/Data/R/functions.R, and is as follows:
    # 
    #   # cohen.w <- function(chisq) {
    #       cramer.v(chisq) * sqrt(min(dim(chisq$observed)-1))
    #   }
    def cohen_w() -> float:
        return cramer_v_val * math.sqrt(
            min(
                contingency_table.shape[0] - 1,
                contingency_table.shape[1] - 1
            )
        )

    cohen_w_val = cohen_w()

    # 4. Inspect symmetrical standardized residuals to identify
    # the cells that contribute to the association, and in which direction.

    # even though I didn't find the original R code
    # of [schaefer_pankratz_2018] for this experiment,
    # the `standardized_resids` property of
    # `statsmodels.stats.contingency_tables.Table`
    # produced the same results as in the paper
    # on the same contingency table, so we can be
    # confident that we are replicating the same analysis
    resids = Table(contingency_table)    \
        .standardized_resids.round(3)
    
    chi2df = pd.Series(
        data={
            "chi2": chi2_res.statistic.round(3),
            "df": chi2_res.dof,
            "p": chi2_res.pvalue,#.round(3),
            "cramer_v": round(cramer_v_val, 3),
            "cohen_w": round(cohen_w_val, 3),
            "resids_cong": resids.loc["l_pl", "num_pl"].item(),
            "resids_incong": resids.loc["l_pl", "num_sg"].item()
        }
    )

    return contingency_table, chi2df

In [24]:
ct_n1, chi2df_n1 = sp18_orig_analysis(ic_alt_nipe_sgl_n1_pll_counts)

ct_n1.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sgl/contingency_table_n1.tsv"),
    **outcsv_args
)

chi2df_n1.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sgl/chi2_n1.tsv"),
    **{**outcsv_args, "header": False}
)

In [29]:
chi2df_n1.round(3)

chi2             3.207
df               1.000
p                0.073
cramer_v         0.045
cohen_w          0.045
resids_cong      1.843
resids_incong   -1.843
dtype: float64

In [15]:
# Compounds built from non-unique constituent combinations (regardless 
# of the linker) appearing only in the Sg form and not undergoing the IntPE
# built from constituent combinations that mostly insert
# a non-pluralic linker in the Sg form
# (N=1333, UniqN=717, UniqCN=490, UniqN1N=46)
ic_alt_nipe_sgl_cc = pd.read_csv(
    os.path.join(sp18_ext_path, "ic_alt_nipe_sgl_cc.tsv"),
    sep="\t",
    header=0,
    index_col="gecodb",
    # all str, WC int, is linker pluralic bool
    dtype=defaultdict(
        lambda: str,
        wc="int",
        is_linker_pl="bool"
    ),
    quoting=3   # = csv.QUOTE_NONE, separate by tabs only
)

ic_alt_nipe_sgl_cc.sample(5)

,const_comb,n1,n2,n1_pl_marker,n1_sem,n2_num,is_linker_pl,wc,context,context_id
gecodb,,,,,,,,,,
stadt_rat,stadt+rat,stadt,rat,_+=e_,ind,pl,False,1132840,Neben Bürgerinitiative und interessierten Einw...,727e998c-4b12-41ca-b84e-ee790fb6e1d5
lied_+er_vortrag,lied+vortrag,lied,vortrag,_+er_,ind,sg,True,248,"""""""Mit einem kurzweiligen Luftballonspiel wurd...",fc714bf7-f3a7-459a-84be-882d5f849093
gitarre_solo,gitarre+solo,gitarre,solo,_+n_,ind,pl,False,0,"Das beste an Ihm ist , dass er immer mitsingt ...",ce388174-8156-4265-9f98-ceca5e477945
haus_eigentümer,haus+eigentümer,haus,eigentümer,_+=er_,ind,pl,False,47348,Neben dem Gas-Brennwertgerät und den Solarkoll...,3c83ead1-0e4a-4eab-aeae-380fbb17bb61
geschenk_buch,geschenk+buch,geschenk,buch,_+e_,ind,pl,False,546,"""""""Ich habe leider überhaupt keine Zeit , mich...",607ba4a5-8b55-4fd9-bdf5-ec6162f9eed9


In [16]:
(
    ic_alt_nipe_sgl_cc_pll_counts,
    ic_alt_nipe_sgl_cc_pll_shares
) = calc_pll_shares(ic_alt_nipe_sgl_cc)

In [ ]:
ic_alt_nipe_sgl_cc_pll_counts.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sgl/pll_counts_cc.tsv"),
    **outcsv_args
)
# (UniqCN=490, UniqN1N=46),
# each constituent combination has 2 rows,
# one for Sg and one for Pl
ic_alt_nipe_sgl_cc_pll_counts

l_sg  l_pl  pl_share
apfel+bauer   num_sg     1     0       0.0
              num_pl     0     1       1.0
apfel+baum    num_sg     2     0       0.0
              num_pl     2     3       0.6
apfel+hof     num_sg     1     0       0.0
...                    ...   ...       ...
zitat+spender num_pl     0     1       1.0
zitat+tag     num_sg     1     0       0.0
              num_pl     1     0       0.0
zitat+zweck   num_sg     2     0       0.0
              num_pl     1     0       0.0

[980 rows x 3 columns]

In [19]:
ic_alt_nipe_sgl_cc_pll_shares.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sgl/pll_shares_cc.tsv"),
    **outcsv_args
)
# (UniqCN=490, UniqN1N=46)
ic_alt_nipe_sgl_cc_pll_shares

,num_sg,num_pl
apfel+bauer,0.00,1.0
apfel+baum,0.00,0.6
apfel+hof,0.00,0.0
apfel+kopf,0.00,1.0
apfel+plantage,0.00,0.5
...,...,...
zitat+gedicht,0.00,0.0
zitat+recht,0.25,0.0
zitat+spender,0.00,1.0
zitat+tag,0.00,0.0


In [20]:
ic_alt_nipe_sgl_cc_pll_shares_desc = ic_alt_nipe_sgl_cc_pll_shares.describe()
ic_alt_nipe_sgl_cc_pll_shares_desc.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sgl/pll_shares_desc_cc.tsv"),
    **outcsv_args
)
ic_alt_nipe_sgl_cc_pll_shares_desc

,num_sg,num_pl
count,490.000000,490.000000
mean,0.109864,0.291361
std,0.202389,0.416000
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.500000
max,0.500000,1.000000


In [26]:
ct_cc, chi2df_cc = sp18_orig_analysis(ic_alt_nipe_sgl_cc_pll_counts)

ct_cc.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sgl/contingency_table_cc.tsv"),
    **outcsv_args
)

chi2df_cc.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sgl/chi2_cc.tsv"),
    **{**outcsv_args, "header": False}
)

In [28]:
chi2df_cc.round(3)

chi2             32.231
df                1.000
p                 0.000
cramer_v          0.155
cohen_w           0.155
resids_cong       5.741
resids_incong    -5.741
dtype: float64